# Episode 16 — HLS: C code to hardware

**High-Level Synthesis (HLS)** turns a C++ function into a hardware block. We write a
31-tap low-pass **FIR filter** in C++ (`hls/fir.cpp`), let Vitis HLS make it into an IP block,
and put it between the two directions of the DMA from episode 15:

```
 DDR --> DMA --> FIR filter (HLS) --> DMA --> DDR
```

The heart of `fir.cpp`:

```cpp
samples: while (!last) {
#pragma HLS PIPELINE II=1                 // a new sample every clock cycle
    axis_t s = x_in.read();
    shift: for (int i = N_TAPS - 1; i > 0; i--) taps[i] = taps[i - 1];   // unrolled
    taps[0] = s.data;
    long long acc = 0;
    mac: for (int i = 0; i < N_TAPS; i++) acc += taps[i] * COEF[i];        // 31 multipliers at once
    ...
}
```

In software the 31 multiplications run one after another. In hardware HLS builds 31 multipliers
that work in parallel, and `PIPELINE II=1` makes the block take a new sample every clock cycle:
**100 million samples a second** at 100 MHz.

## Building

1. `vitis_hls -f run_hls.tcl` in the `hls` folder: C simulation, synthesis, export as IP.
2. `vivado -mode batch -source vivado/fir_overlay.tcl -tclargs xc7z020clg400-1 work overlays/z20 hls/fir_prj/solution1/impl/ip`

In [ ]:
from pynq import Overlay, allocate
from pynq.overlays.base import board_variant
import numpy as np
import time
%matplotlib inline
import matplotlib.pyplot as plt

chip = {"EDGE-Z10": "z10", "EDGE-Z20": "z20"}[board_variant()]
ol = Overlay("overlays/{}/fir_overlay.bit".format(chip))
dma = ol.axi_dma
fir = ol.fir_0

## 1. Start the HLS block

An HLS block has a control register at offset 0: bit 0 **ap_start** starts it, and bit 7
**auto_restart** makes it start again by itself after each run. With both set, the filter
handles every DMA transfer we send.

In [ ]:
fir.write(0x00, 0x81)
print("control register: 0x{:02X}".format(fir.read(0x00)))

## 2. A test signal

A slow sine (which the filter should keep), a fast sine (which it should remove) and some noise,
as 16-bit samples. Each sample travels in a 32-bit word.

In [ ]:
fs = 1_000_000                               # pretend sample rate: 1 MHz
n = 100_000
t = np.arange(n) / fs
signal = (9000 * np.sin(2 * np.pi * 10_000 * t)            # 10 kHz: keep
          + 9000 * np.sin(2 * np.pi * 300_000 * t)         # 300 kHz: remove
          + np.random.normal(0, 1500, n))
x = np.clip(signal, -32768, 32767).astype(np.int16)

def fir_hardware(samples):
    """Filter int16 samples in the PL."""
    inp = allocate(shape=(len(samples),), dtype=np.int32)
    out = allocate(shape=(len(samples),), dtype=np.int32)
    inp[:] = samples
    dma.recvchannel.transfer(out)
    dma.sendchannel.transfer(inp)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    result = np.array(out, dtype=np.int16)
    inp.freebuffer()
    out.freebuffer()
    return result

y = fir_hardware(x)

plt.figure(figsize=(10, 3))
plt.plot(t[:300] * 1e6, x[:300], label="input", alpha=0.6)
plt.plot(t[:300] * 1e6, y[:300], label="FIR output", linewidth=2)
plt.xlabel("time (µs)")
plt.legend()
plt.show()

## 3. Is the hardware right?

The same filter in NumPy, with the same integer arithmetic (coefficients scaled by 2^15,
rounding, saturation to 16 bits). The two must agree exactly.

In [ ]:
COEF = np.array([53, 46, 24, -36, -150, -306, -451, -492, -321, 148, 941, 1993, 3154, 4214, 4957,
                 5220, 4957, 4214, 3154, 1993, 941, 148, -321, -492, -451, -306, -150, -36, 24, 46, 53])

def fir_numpy(samples):
    acc = np.convolve(samples.astype(np.int64), COEF)[:len(samples)]
    return np.clip((acc + (1 << 14)) >> 15, -32768, 32767).astype(np.int16)

reference = fir_numpy(x)
print("samples that differ:", int(np.count_nonzero(reference != y)), "of", n)

## 4. The spectrum

In [ ]:
def spectrum_db(v):
    s = np.abs(np.fft.rfft(v * np.hanning(len(v))))
    return 20 * np.log10(s / s.max() + 1e-12)

f = np.fft.rfftfreq(n, 1 / fs) / 1000
plt.figure(figsize=(10, 3))
plt.plot(f, spectrum_db(x), label="input", alpha=0.6)
plt.plot(f, spectrum_db(y), label="FIR output")
plt.xlabel("frequency (kHz)")
plt.ylabel("dB")
plt.ylim(-100, 5)
plt.legend()
plt.show()

## 5. Hardware against software

One million samples through the FPGA, through NumPy (optimised C on the ARM), and through a
plain Python loop (timed on 2 000 samples and scaled up, because the full run takes minutes).

In [ ]:
big = np.clip(np.random.normal(0, 8000, 1_000_000), -32768, 32767).astype(np.int16)

t0 = time.perf_counter(); y_hw = fir_hardware(big); t_hw = time.perf_counter() - t0
t0 = time.perf_counter(); y_np = fir_numpy(big);    t_np = time.perf_counter() - t0

def fir_python(samples):
    out = []
    for k in range(len(samples)):
        acc = 0
        for i in range(31):
            if k - i >= 0:
                acc += int(samples[k - i]) * int(COEF[i])
        out.append(max(-32768, min(32767, (acc + (1 << 14)) >> 15)))
    return out

t0 = time.perf_counter(); fir_python(big[:2000]); t_py = (time.perf_counter() - t0) * len(big) / 2000

print("same result:", np.array_equal(y_hw, y_np))
for name, sec in [("FPGA (HLS + DMA)", t_hw), ("NumPy", t_np), ("Python loop", t_py)]:
    print("{:18s} {:9.3f} s   {:7.1f} x the FPGA time".format(name, sec, sec / t_hw))

Most of the FPGA time is copying the samples into and out of the DMA buffers in Python;
the filter itself needs 10 ms for a million samples.

## Try it yourself

Change `COEF` in `fir.h` to a **high-pass** filter (keep the 300 kHz sine, remove the 10 kHz one),
rebuild with Vitis HLS and Vivado, and update `COEF` in this notebook to match.